# 🔬 Microplastic Detection — YOLO26m Training (Kaggle · GPU T4 ×2)

Trains **`yolo26m`** on the *microplastic_yolo26_training_dataset* (MINA Lab, Univ. of Dhaka + community data · 9,238 images) with strengthened augmentation.

End-to-end pipeline: **install → verify GPU → build data.yaml → train → validate → export (ONNX + PyTorch) → test inference → package for website**.

**Before running:** Notebook settings (right panel) →
- **Accelerator: GPU T4 x2**  ⭐ *(use T4, not P100 — Kaggle's current PyTorch 2.10+cu128 dropped Pascal/P100 support, so P100 errors with 'no kernel image'. T4 is Turing and fully supported.)*  Not TPU either — Ultralytics YOLO runs on PyTorch/CUDA, which TPU doesn't support.
- **Internet: ON**  (needed to pip-install ultralytics and download pretrained weights)
- **Add data:** attach the dataset you uploaded (folder with `train/ valid/ test/`).


> 🛑 **RUN THIS AS A COMMIT — do NOT sit on the interactive tab.** Training takes ~8 h. In an interactive session Kaggle **wipes `/kaggle/working` when the kernel goes idle (~20–40 min) or the tab closes** — you lose the trained weights. Instead click **Save Version → Save & Run All (Commit)**. Kaggle then runs the whole notebook **headless** (up to 12 h) and saves everything in `/kaggle/working` as the version's **Output**, which is permanent and downloadable afterward. It cannot be lost to idle timeout because there's no interactive session to time out. (Headless is also more reliable for multi-GPU DDP.)

**Before running:** Accelerator **GPU T4 x2**, Internet **ON**, attach your dataset. Then **Save & Run All (Commit)** and walk away — check the finished version's Output for `microplastic_model_package.zip`.

> ⚠️ **P100 note:** Kaggle's current image ships PyTorch 2.10+cu128, which has **no P100 (Pascal) kernels** — training on P100 fails with *'no kernel image is available'*. **Use GPU T4 x2.**

## 1 · Environment & GPU check

In [ ]:
import subprocess, torch, platform
print('Python :', platform.python_version())
print('Torch  :', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
    print('VRAM   : %.1f GB' % (torch.cuda.get_device_properties(0).total_memory/1e9))
else:
    print('⚠️  No GPU detected — set Accelerator = GPU P100 in notebook settings.')
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

## 2 · Install Ultralytics (YOLO26) — **without breaking Kaggle's P100 PyTorch**
`pip install -U ultralytics` pulls a newer torch whose CUDA build has **no Pascal (sm_60) kernels**, which crashes the P100 with *'no kernel image is available for execution on the device'*. We upgrade **only ultralytics** (`--no-deps`) and keep Kaggle's working torch.

In [ ]:
import torch
print('Preinstalled torch:', torch.__version__, '| CUDA build:', torch.version.cuda)

# -U upgrades ultralytics to a YOLO26-capable version; --no-deps protects torch/torchvision.
!pip install -q -U ultralytics --no-deps
!pip install -q ultralytics-thop py-cpuinfo   # tiny pure-python deps, safe to add

import ultralytics, importlib
importlib.reload(ultralytics)
print('ultralytics', ultralytics.__version__)

# Smoke-test that THIS torch can actually run kernels on the P100 before training:
_x = torch.zeros(1, device='cuda') + 1
print('✅ CUDA kernel test passed on', torch.cuda.get_device_name(0))

## 3 · Locate the uploaded dataset (auto-detect)

In [ ]:
import glob, os
# Find the folder that contains train/images anywhere under /kaggle/input.
# This is robust to however the dataset zip was extracted.
cands = glob.glob('/kaggle/input/**/train/images', recursive=True)
assert cands, 'Could not find */train/images under /kaggle/input — did you attach the dataset?'
DATA_ROOT = os.path.dirname(os.path.dirname(cands[0]))
print('Dataset root:', DATA_ROOT)
for sp in ['train', 'valid', 'test']:
    ip = os.path.join(DATA_ROOT, sp, 'images')
    n = len(glob.glob(os.path.join(ip, '*'))) if os.path.isdir(ip) else 0
    print(f'  {sp:5s}: {n} images')

## 4 · Write the Kaggle `data.yaml`
The uploaded `data.yaml` points at a Windows path, so we regenerate it with the Kaggle mount path.

In [ ]:
data_yaml_path = '/kaggle/working/data.yaml'
yaml_text = f'''# Microplastic YOLO26 — Kaggle runtime config
path: {DATA_ROOT}
train: train/images
val: valid/images
test: test/images

nc: 4
names:
  0: fiber
  1: film
  2: fragment
  3: pellet
'''
with open(data_yaml_path, 'w') as f:
    f.write(yaml_text)
print(yaml_text)

## 5 · Training configuration + augmentation
**Model: `yolo26m`** (medium — more capacity than the `s` baseline) with **strengthened augmentation** for robustness to lighting, colour-cast and orientation.

**Defaults below = GPU T4 ×2** (`DEVICE=[0,1]`, `BATCH=32`, DDP across both cards). This is the proven config: `yolo26m` @ 640 ≈ 3.4 min/epoch → **150 epochs ≈ 8.5 h**, comfortably inside the 12 h limit. On a single T4 instead, set `DEVICE=0`, `BATCH=16`.

**Accuracy knob — resolution.** Microplastics are small, so **input resolution is the single highest-impact lever**. `imgsz=640` is the safe default; for a further push set **`IMGSZ=768`** and drop **`EPOCHS≈120`** (≈768²/640² more compute per epoch → still ~10 h on T4 ×2). Expect the biggest gains on the `pellet` class and on mAP50-95. If you hit OOM at 768, lower `BATCH` to 24.

Everything else (`cos_lr`, strengthened `AUG`) is set for best final-epoch convergence. If in-domain mAP ever drops vs baseline, ease `hsv_*` / `mixup` back toward defaults.

In [ ]:
MODEL     = 'yolo26m.pt'   # n < s < m < l < x   (accuracy up, speed down)
EPOCHS    = 150            # ~8.5h on T4 x2 @ 640. If IMGSZ=768, use ~120.
IMGSZ     = 640            # dataset has 480/512/640px imgs. Accuracy push: 768 (see notes)
BATCH     = 32             # T4 x2 = 16/GPU @ 640. Single T4 -> 16. OOM -> halve it
PATIENCE  = 40             # early-stop if val mAP plateaus (saves time within 12h)
SAVE_PERIOD = 25           # write a checkpoint every N epochs (safety net; -1 disables)
DEVICE    = [0, 1]         # GPU T4 x2 (DDP). Single GPU -> DEVICE = 0
WORKERS   = 4              # Kaggle gives ~4 vCPUs
SEED      = 42
RESUME    = False          # set True to continue a previous run (see notes at bottom)
PROJECT   = '/kaggle/working/microplastic'
RUN_NAME  = 'yolo26m_aug_run'  # new name — keeps earlier runs intact for comparison

# ── Strengthened augmentation (robustness to lighting / colour / orientation) ──
#    [def X] = Ultralytics default the baseline model used.
AUG = dict(
    hsv_h=0.05,       # hue jitter — colour-cast robustness (warm/amber tints)   [def 0.015]
    hsv_s=0.80,       # saturation jitter                                         [def 0.70]
    hsv_v=0.50,       # value/brightness jitter — lighting robustness             [def 0.40]
    degrees=15.0,     # rotation — microscopy has no canonical 'up'               [def 0.0]
    translate=0.10,   # shift                                                     [def 0.10]
    scale=0.60,       # zoom — magnification robustness                           [def 0.50]
    fliplr=0.50,      # horizontal flip                                           [def 0.50]
    flipud=0.50,      # vertical flip — valid for orientation-free microscopy     [def 0.0]
    mosaic=1.00,      # 4-image mosaic                                            [def 1.0]
    mixup=0.15,       # image blend — extra regularisation                       [def 0.0]
    close_mosaic=15,  # disable mosaic for the final epochs (clean convergence)   [def 10]
)

## 6 · Train

In [ ]:
from ultralytics import YOLO
import os

last_ckpt = os.path.join(PROJECT, RUN_NAME, 'weights', 'last.pt')
if RESUME and os.path.exists(last_ckpt):
    print('Resuming from', last_ckpt)
    model = YOLO(last_ckpt)
    resume_flag = True
else:
    try:
        model = YOLO(MODEL)                 # auto-downloads pretrained YOLO26 weights
    except Exception as e:
        print('YOLO26 weights unavailable (', e, ') — falling back to yolo11s.pt')
        model = YOLO('yolo11s.pt')
    resume_flag = False

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    save_period=SAVE_PERIOD,   # periodic checkpoints (recoverable in a Commit run)
    workers=WORKERS,
    seed=SEED,
    device=DEVICE,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    resume=resume_flag,
    cache=False,           # 9k imgs — avoid RAM cache OOM on Kaggle
    amp=True,              # mixed precision — faster on T4 tensor cores
    cos_lr=True,           # cosine LR decay — smoother final-epoch convergence
    plots=True,            # save curves + confusion matrix
    **AUG,                 # ← strengthened augmentation (defined in cell 5)
)

# Derive the output dir from PROJECT/RUN_NAME (deterministic with exist_ok=True).
# NOTE: under multi-GPU DDP (device=[0,1]) model.train() returns a plain dict with
# no .save_dir, so we must NOT rely on results.save_dir here.
SAVE_DIR = os.path.join(PROJECT, RUN_NAME)
assert os.path.exists(os.path.join(SAVE_DIR, 'weights', 'best.pt')), 'best.pt not found in ' + SAVE_DIR
print('Run saved to:', SAVE_DIR)

## 7 · Validate on the held-out TEST split
Reports mAP50, mAP50-95, precision/recall — overall and per class.

In [ ]:
best = os.path.join(SAVE_DIR, 'weights', 'best.pt')
best_model = YOLO(best)
metrics = best_model.val(data=data_yaml_path, split='test', imgsz=IMGSZ, workers=WORKERS)
print('\n================ TEST METRICS ================')
print(f'mAP50-95 : {metrics.box.map:.4f}')
print(f'mAP50    : {metrics.box.map50:.4f}')
print(f'mAP75    : {metrics.box.map75:.4f}')
names = metrics.names
print('\nPer-class AP50:')
try:
    for ci, ap in zip(metrics.box.ap_class_index, metrics.box.ap50):
        print(f'  {names[int(ci)]:9s}  AP50={ap:.4f}')
except Exception as e:
    print('per-class print skipped:', e)

## 8 · Export for the website (ONNX + PyTorch)
YOLO26 is **NMS-free / end-to-end**, so the ONNX graph already outputs final decoded boxes — ideal for browser (`onnxruntime-web`) or a Python backend (`onnxruntime`).

In [ ]:
import shutil
OUT = '/kaggle/working'
# 1) PyTorch weight (use with `from ultralytics import YOLO`)
shutil.copy(best, os.path.join(OUT, 'microplastic_yolo26_best.pt'))

# 2) ONNX (portable inference)
onnx_path = None
try:
    onnx_path = best_model.export(format='onnx', imgsz=IMGSZ, opset=13, simplify=True, dynamic=False)
    shutil.copy(onnx_path, os.path.join(OUT, 'microplastic_yolo26.onnx'))
    print('ONNX exported:', onnx_path)
except Exception as e:
    print('ONNX export failed:', e)

# 3) Copy training plots for the report
for fn in ['results.png','confusion_matrix.png','confusion_matrix_normalized.png',
           'PR_curve.png','labels.jpg','args.yaml','results.csv']:
    src = os.path.join(SAVE_DIR, fn)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(OUT, fn))
print('Copied artifacts to', OUT)

## 9 · Sanity-check: run inference on sample TEST images

In [ ]:
import glob, matplotlib.pyplot as plt, cv2, random
test_imgs = glob.glob(os.path.join(DATA_ROOT, 'test', 'images', '*'))
random.seed(0)
sample = random.sample(test_imgs, min(6, len(test_imgs)))
preds = best_model.predict(sample, imgsz=IMGSZ, conf=0.25, save=False)
plt.figure(figsize=(16, 10))
for i, r in enumerate(preds):
    im = cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)
    plt.subplot(2, 3, i+1); plt.imshow(im); plt.axis('off')
    plt.title(os.path.basename(r.path)[:22], fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT,'sample_predictions.png'), dpi=120); plt.show()

## 10 · Package everything into one downloadable ZIP

In [ ]:
import shutil, os
pkg_dir = '/kaggle/working/microplastic_model_package'
os.makedirs(pkg_dir, exist_ok=True)
for f in ['microplastic_yolo26_best.pt','microplastic_yolo26.onnx','data.yaml',
          'results.png','confusion_matrix.png','PR_curve.png','results.csv',
          'sample_predictions.png','args.yaml']:
    p = os.path.join('/kaggle/working', f)
    if os.path.exists(p): shutil.copy(p, pkg_dir)
zip_path = shutil.make_archive('/kaggle/working/microplastic_model_package', 'zip', pkg_dir)
print('✅ Download this from the Output tab →', zip_path)
print('\nContents:')
for f in sorted(os.listdir(pkg_dir)):
    print('  ', f, f'({os.path.getsize(os.path.join(pkg_dir,f))/1e6:.2f} MB)')

## 11 · How to use the trained model in your website

**Option A — Python backend (FastAPI/Flask), simplest & most accurate:**
```python
from ultralytics import YOLO
model = YOLO('microplastic_yolo26_best.pt')       # or the .onnx
res = model.predict('upload.jpg', conf=0.25, imgsz=640)
for b in res[0].boxes:
    cls = res[0].names[int(b.cls)]; conf = float(b.conf)
    x1,y1,x2,y2 = b.xyxy[0].tolist()
    print(cls, round(conf,3), [round(v) for v in (x1,y1,x2,y2)])
```

**Option B — In-browser, no server (`onnxruntime-web`):** ship `microplastic_yolo26.onnx` (YOLO26 is NMS-free, so the model output is already the final boxes — you only threshold by confidence and scale coords back to the original image size). Classes are `['fiber','film','fragment','pellet']`.

**Option C — ONNX in a Python backend (no ultralytics dep):** run with `onnxruntime`, letterbox to 640×640, feed NCHW float32 `/255`, read decoded boxes from the output tensor.

---
### If the 12 h session isn't enough / it times out
1. Use **Save Version → Save & Run All (Commit)** — runs headless up to 12 h and saves `/kaggle/working`.
2. To **resume**: add this run's output as an input dataset, copy `runs` back, set `RESUME=True`, re-run. Or just lower `EPOCHS` — with early stopping (`patience=40`) the model usually converges well before 150 epochs on this dataset.